In [5]:
"""
=============================================================
  TELECOMX - ANÁLISIS DE EVASIÓN DE CLIENTES (CHURN)
=============================================================
  Pasos:
  1. Carga de datos desde API (JSON)
  2. Exploración del dataset
  3. Identificación de problemas en los datos
  4. Limpieza y corrección de datos
  5. Creación de columna "Cuentas_Diarias"
  6. Estandarización y transformación de datos
=============================================================
"""

import pandas as pd
import numpy as np
import json
# ─────────────────────────────────────────────────────────────
# PASO 1 — CARGA DE DATOS DESDE LA API
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("PASO 1: CARGA DE DATOS DESDE LA API")
print("=" * 60)

ARCHIVO = "/content/TelecomX_Data.json"

with open(ARCHIVO, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"✅ Datos cargados correctamente. Total de registros: {len(raw_data)}")
print(f"   Tipo de objeto recibido: {type(raw_data)}")
print(f"   Ejemplo de un registro (keys): {list(raw_data[0].keys())}")

# Aplanar el JSON anidado a un DataFrame plano
df = pd.json_normalize(raw_data)

print(f"\n✅ DataFrame creado. Shape: {df.shape}")
print(f"   Filas: {df.shape[0]} | Columnas: {df.shape[1]}")


# ─────────────────────────────────────────────────────────────
# PASO 2 — EXPLORACIÓN DEL DATASET
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PASO 2: EXPLORACIÓN DEL DATASET")
print("=" * 60)

print("\n📋 Columnas del dataset:")
for col in df.columns:
    print(f"   • {col}")

print("\n📊 Tipos de datos por columna:")
print(df.dtypes.to_string())

print("\n📈 Estadísticas descriptivas (columnas numéricas):")
print(df.describe().to_string())

print("\n📌 Primeras 5 filas:")
print(df.head().to_string())

print("\n📌 Diccionario de variables relevantes para análisis de evasión:")
relevantes = {
    "customerID":              "ID único del cliente",
    "Churn":                   "Variable objetivo — indica si el cliente abandonó",
    "customer.tenure":         "Meses que lleva el cliente con la empresa",
    "customer.SeniorCitizen":  "Si tiene 65 años o más (0/1)",
    "account.Contract":        "Tipo de contrato (mes a mes, 1 año, 2 años)",
    "account.PaymentMethod":   "Método de pago",
    "account.Charges.Monthly": "Cargo mensual total",
    "account.Charges.Total":   "Total facturado histórico",
    "internet.InternetService":"Tipo de servicio de internet",
}
for k, v in relevantes.items():
    print(f"   {k:<32} → {v}")


# ─────────────────────────────────────────────────────────────
# PASO 3 — IDENTIFICACIÓN DE PROBLEMAS EN LOS DATOS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PASO 3: IDENTIFICACIÓN DE PROBLEMAS EN LOS DATOS")
print("=" * 60)

# 3.1 Valores nulos + strings vacíos en una sola pasada
# pandas.unique() permite inspeccionar categorías sin crear objetos intermedios
print("\n🔍 3.1 — Nulos y valores vacíos por columna:")
for col in df.columns:
    nulos   = df[col].isnull().sum()
    # pd.unique() es más eficiente que .value_counts(); detectamos '' entre los valores únicos
    vacios  = ("" in pd.unique(df[col].astype(str).str.strip()))
    n_vacios = (df[col].astype(str).str.strip() == "").sum() if vacios else 0
    if nulos > 0 or n_vacios > 0:
        print(f"   ⚠️  '{col}': {nulos} nulos | {n_vacios} vacíos")
print("   ✅ Revisión completada.")

# 3.2 Duplicados
print("\n🔍 3.2 — Filas duplicadas:")
duplicados = df.duplicated().sum()
print(f"   {'⚠️  ' if duplicados > 0 else '✅ '}Duplicados encontrados: {duplicados}")

# 3.3 Tipos de datos y categorías únicas con pandas.unique()
# pd.unique() recorre el array una sola vez (O(n)), sin ordenar — más rápido que .unique() sobre Series
print("\n🔍 3.3 — Tipos de datos y valores únicos por columna:")
for col in df.columns:
    dtype   = df[col].dtype
    unicos  = pd.unique(df[col])        # ← tip: pd.unique() directamente sobre la columna
    n_unicos = len(unicos)
    # Mostrar detalle si es categórica (pocos valores) o alertar si el tipo es sospechoso
    if n_unicos <= 10:
        print(f"   {col:<35} dtype={str(dtype):<10} únicos({n_unicos}): {unicos.tolist()}")
    else:
        alerta = " ⚠️  (debería ser numérico)" if col == "account.Charges.Total" else ""
        print(f"   {col:<35} dtype={str(dtype):<10} únicos: {n_unicos}{alerta}")


# ─────────────────────────────────────────────────────────────
# PASO 4 — LIMPIEZA Y CORRECCIÓN DE DATOS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PASO 4: LIMPIEZA Y CORRECCIÓN DE DATOS")
print("=" * 60)

df_clean = df.copy()

# 4.1 Convertir 'Charges.Total' a numérico
df_clean['account.Charges.Total'] = pd.to_numeric(
    df_clean['account.Charges.Total'], errors='coerce'
)
print(f"✅ 'account.Charges.Total' convertido a numérico.")

# 4.2 Imputar NaN generados en Charges.Total con la mediana
mediana_total = df_clean['account.Charges.Total'].median()
nulos_total = df_clean['account.Charges.Total'].isna().sum()
df_clean['account.Charges.Total'] = df_clean['account.Charges.Total'].fillna(mediana_total)
print(f"✅ {nulos_total} valores nulos en 'Charges.Total' imputados con la mediana ({mediana_total:.2f}).")

# 4.2b Tratar Churn vacío como NaN y eliminar esas filas
churn_vacios = (df_clean['Churn'].astype(str).str.strip() == "").sum()
df_clean = df_clean[df_clean['Churn'].astype(str).str.strip() != ""].reset_index(drop=True)
print(f"✅ {churn_vacios} filas con 'Churn' vacío eliminadas (sin variable objetivo).")

# 4.3 Eliminar duplicados
antes = len(df_clean)
df_clean.drop_duplicates(inplace=True)
df_clean.reset_index(drop=True, inplace=True)
print(f"✅ Duplicados eliminados: {antes - len(df_clean)} filas removidas.")

# 4.4 Eliminar filas con customerID nulo
nulos_id = df_clean['customerID'].isna().sum()
df_clean.dropna(subset=['customerID'], inplace=True)
print(f"✅ Filas sin customerID eliminadas: {nulos_id}")

# 4.5 Estandarizar espacios en strings
str_cols = df_clean.select_dtypes(include="object").columns
for col in str_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()
print(f"✅ Espacios en blanco estandarizados en {len(str_cols)} columnas de texto.")

print(f"\n📊 Shape final después de limpieza: {df_clean.shape}")
print(f"   Nulos restantes: {df_clean.isnull().sum().sum()}")


# ─────────────────────────────────────────────────────────────
# PASO 5 — CREACIÓN DE COLUMNA "Cuentas_Diarias"
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PASO 5: CREACIÓN DE LA COLUMNA 'Cuentas_Diarias'")
print("=" * 60)

# Cargo mensual / 30 días = cargo diario estimado
df_clean['Cuentas_Diarias'] = (df_clean['account.Charges.Monthly'] / 30).round(4)

print("✅ Columna 'Cuentas_Diarias' creada: Charges.Monthly / 30 días")
print("\n📊 Estadísticas de 'Cuentas_Diarias':")
print(df_clean['Cuentas_Diarias'].describe().to_string())
print(f"\n   Muestra (primeros 5 registros):")
print(df_clean[['customerID', 'account.Charges.Monthly', 'Cuentas_Diarias']].head().to_string())


# ─────────────────────────────────────────────────────────────
# PASO 6 — ESTANDARIZACIÓN Y TRANSFORMACIÓN DE DATOS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PASO 6: ESTANDARIZACIÓN Y TRANSFORMACIÓN DE DATOS")
print("=" * 60)

df_final = df_clean.copy()

# 6.1 Mapear columnas binarias Yes/No → 1/0
cols_binarias = [
    "customer.Partner", "customer.Dependents",
    "phone.PhoneService", "account.PaperlessBilling", "Churn"
]
mapa_binario = {"Yes": 1, "No": 0, "yes": 1, "no": 0}
for col in cols_binarias:
    if col in df_final.columns:
        df_final[col] = df_final[col].map(mapa_binario)
        print(f"   ✅ '{col}' → 1/0")

# 6.2 Mapear SeniorCitizen (ya es 0/1, solo verificar)
print(f"   ✅ 'customer.SeniorCitizen' ya es numérico: {df_final['customer.SeniorCitizen'].unique()}")

# 6.3 Renombrar columnas al español para mayor claridad
renombrar = {
    "customerID":                    "ID_Cliente",
    "Churn":                         "Evasion",
    "customer.gender":               "Genero",
    "customer.SeniorCitizen":        "Adulto_Mayor",
    "customer.Partner":              "Tiene_Pareja",
    "customer.Dependents":           "Tiene_Dependientes",
    "customer.tenure":               "Meses_Contrato",
    "phone.PhoneService":            "Servicio_Telefono",
    "phone.MultipleLines":           "Multiples_Lineas",
    "internet.InternetService":      "Tipo_Internet",
    "internet.OnlineSecurity":       "Seguridad_Online",
    "internet.OnlineBackup":         "Respaldo_Online",
    "internet.DeviceProtection":     "Proteccion_Dispositivo",
    "internet.TechSupport":          "Soporte_Tecnico",
    "internet.StreamingTV":          "Streaming_TV",
    "internet.StreamingMovies":      "Streaming_Peliculas",
    "account.Contract":              "Tipo_Contrato",
    "account.PaperlessBilling":      "Factura_Digital",
    "account.PaymentMethod":         "Metodo_Pago",
    "account.Charges.Monthly":       "Cargo_Mensual",
    "account.Charges.Total":         "Cargo_Total",
}
df_final.rename(columns=renombrar, inplace=True)
print(f"\n✅ Columnas renombradas al español.")

# 6.4 Traducir valores categóricos de inglés a español
traducciones = {
    "Genero":        {"Female": "Femenino", "Male": "Masculino"},
    "Tipo_Internet": {"DSL": "DSL", "Fiber optic": "Fibra_Optica", "No": "Sin_Internet"},
    "Tipo_Contrato": {
        "Month-to-month": "Mes_a_Mes",
        "One year":       "Un_Anio",
        "Two year":       "Dos_Anios",
    },
    "Metodo_Pago": {
        "Electronic check":             "Cheque_Electronico",
        "Mailed check":                 "Cheque_Correo",
        "Bank transfer (automatic)":    "Transferencia_Bancaria",
        "Credit card (automatic)":      "Tarjeta_Credito",
    },
}
for col, mapa in traducciones.items():
    if col in df_final.columns:
        df_final[col] = df_final[col].map(mapa).fillna(df_final[col])
        print(f"   ✅ '{col}' traducido al español.")

# 6.5 Vista final
print(f"\n📊 Dataset final — Shape: {df_final.shape}")
print(f"\n📋 Columnas finales:")
for col in df_final.columns:
    print(f"   • {col} ({df_final[col].dtype})")

print(f"\n📌 Primeras 3 filas del dataset final:")
print(df_final.head(3).to_string())

# Guardar resultado
df_final.to_csv("/content/TelecomX_limpio.csv", index=False)
print("\n✅ Dataset limpio guardado en: TelecomX_limpio.csv")

print("\n" + "=" * 60)
print("✅ TODOS LOS PASOS COMPLETADOS EXITOSAMENTE")
print("=" * 60)

PASO 1: CARGA DE DATOS DESDE LA API
✅ Datos cargados correctamente. Total de registros: 7267
   Tipo de objeto recibido: <class 'list'>
   Ejemplo de un registro (keys): ['customerID', 'Churn', 'customer', 'phone', 'internet', 'account']

✅ DataFrame creado. Shape: (7267, 21)
   Filas: 7267 | Columnas: 21

PASO 2: EXPLORACIÓN DEL DATASET

📋 Columnas del dataset:
   • customerID
   • Churn
   • customer.gender
   • customer.SeniorCitizen
   • customer.Partner
   • customer.Dependents
   • customer.tenure
   • phone.PhoneService
   • phone.MultipleLines
   • internet.InternetService
   • internet.OnlineSecurity
   • internet.OnlineBackup
   • internet.DeviceProtection
   • internet.TechSupport
   • internet.StreamingTV
   • internet.StreamingMovies
   • account.Contract
   • account.PaperlessBilling
   • account.PaymentMethod
   • account.Charges.Monthly
   • account.Charges.Total

📊 Tipos de datos por columna:
customerID                    object
Churn                         object
cus

In [3]:
"""
=============================================================
  TELECOMX — ANÁLISIS EXPLORATORIO DE DATOS (EDA)
=============================================================
  7.  Análisis descriptivo
  8.  Distribución de la variable Evasión (Churn)
  9.  Evasión por variables categóricas
  10. Evasión por variables numéricas
=============================================================
"""

import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Estilo global ──────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.05)
AZUL   = "#4C9BE8"   # clientes que permanecen
ROJO   = "#E8634C"   # clientes que evaden
PALETA = [AZUL, ROJO]
OUT    = "/content/grafico"

# ── Reconstruir df_final desde el JSON (reproduce pasos 1-6) ─
ARCHIVO = "/content/TelecomX_Data.json"
with open(ARCHIVO, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.json_normalize(raw_data)

# Limpieza básica (Paso 4)
df["account.Charges.Total"] = pd.to_numeric(df["account.Charges.Total"], errors="coerce")
mediana = df["account.Charges.Total"].median()
df["account.Charges.Total"] = df["account.Charges.Total"].fillna(mediana)
df = df[df["Churn"].astype(str).str.strip() != ""].reset_index(drop=True)

# Cuentas_Diarias (Paso 5)
df["Cuentas_Diarias"] = (df["account.Charges.Monthly"] / 30).round(4)

# Estandarización (Paso 6)
mapa_bin = {"Yes": 1, "No": 0}
for col in ["customer.Partner", "customer.Dependents",
            "phone.PhoneService", "account.PaperlessBilling"]:
    df[col] = df[col].map(mapa_bin)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df.rename(columns={
    "customerID":               "ID_Cliente",
    "Churn":                    "Evasion",
    "customer.gender":          "Genero",
    "customer.SeniorCitizen":   "Adulto_Mayor",
    "customer.Partner":         "Tiene_Pareja",
    "customer.Dependents":      "Tiene_Dependientes",
    "customer.tenure":          "Meses_Contrato",
    "phone.PhoneService":       "Servicio_Telefono",
    "phone.MultipleLines":      "Multiples_Lineas",
    "internet.InternetService": "Tipo_Internet",
    "internet.OnlineSecurity":  "Seguridad_Online",
    "internet.OnlineBackup":    "Respaldo_Online",
    "internet.DeviceProtection":"Proteccion_Dispositivo",
    "internet.TechSupport":     "Soporte_Tecnico",
    "internet.StreamingTV":     "Streaming_TV",
    "internet.StreamingMovies": "Streaming_Peliculas",
    "account.Contract":         "Tipo_Contrato",
    "account.PaperlessBilling": "Factura_Digital",
    "account.PaymentMethod":    "Metodo_Pago",
    "account.Charges.Monthly":  "Cargo_Mensual",
    "account.Charges.Total":    "Cargo_Total",
}, inplace=True)

traducciones = {
    "Genero":        {"Female": "Femenino", "Male": "Masculino"},
    "Tipo_Internet": {"Fiber optic": "Fibra Óptica", "No": "Sin Internet"},
    "Tipo_Contrato": {"Month-to-month": "Mes a Mes",
                      "One year": "Un Año", "Two year": "Dos Años"},
    "Metodo_Pago":   {"Electronic check":          "Cheque Electrónico",
                      "Mailed check":              "Cheque Correo",
                      "Bank transfer (automatic)": "Transferencia Bancaria",
                      "Credit card (automatic)":   "Tarjeta de Crédito"},
}
for col, mapa in traducciones.items():
    df[col] = df[col].map(mapa).fillna(df[col])

# Etiqueta legible para la variable objetivo
df["Evasion_Label"] = df["Evasion"].map({1: "Si", 0: "No"})

print(f"✅ Dataset listo — {df.shape[0]} filas × {df.shape[1]} columnas\n")


# ══════════════════════════════════════════════════════════════
# PASO 7 — ANÁLISIS DESCRIPTIVO
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("PASO 7: ANÁLISIS DESCRIPTIVO")
print("=" * 60)

COLS_NUM = ["Meses_Contrato", "Cargo_Mensual", "Cargo_Total", "Cuentas_Diarias"]

# describe() con percentiles extendidos
desc = df[COLS_NUM].describe(percentiles=[.25, .50, .75, .90])
print("\n📊 Estadísticas generales (DataFrame.describe()):")
print(desc.round(2).to_string())

# Métricas adicionales por variable
print("\n📊 Métricas adicionales:")
for col in COLS_NUM:
    serie = df[col]
    print(f"\n  {col}:")
    print(f"    Media:          {serie.mean():.2f}")
    print(f"    Mediana:        {serie.median():.2f}")
    print(f"    Desv. Estándar: {serie.std():.2f}")
    print(f"    Asimetría:      {serie.skew():.2f}")
    print(f"    Curtosis:       {serie.kurt():.2f}")
    print(f"    Rango:          {serie.min():.2f} – {serie.max():.2f}")

# ── Gráfico: distribución de las 4 variables numéricas ────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Distribución de Variables Numéricas", fontsize=15, fontweight="bold", y=1.01)

for ax, col in zip(axes.flat, COLS_NUM):
    media   = df[col].mean()
    mediana = df[col].median()
    sns.histplot(df[col], ax=ax, color=AZUL, bins=35, kde=True, edgecolor="white", linewidth=0.4)
    ax.axvline(media,   color="#E8634C", linestyle="--", linewidth=1.4, label=f"Media {media:.1f}")
    ax.axvline(mediana, color="#2e7d32", linestyle=":",  linewidth=1.4, label=f"Mediana {mediana:.1f}")
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("")
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(f"{OUT}paso7_distribucion_numericas.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\n✅ Gráfico guardado: paso7_distribucion_numericas.png")


# ══════════════════════════════════════════════════════════════
# PASO 8 — DISTRIBUCIÓN DE LA VARIABLE EVASIÓN
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 8: DISTRIBUCIÓN DE LA VARIABLE EVASIÓN")
print("=" * 60)

conteo   = df["Evasion_Label"].value_counts()
pct      = df["Evasion_Label"].value_counts(normalize=True) * 100

print("\n📊 Conteo y proporción de Evasión:")
for label in ["No", "Si"]:
    print(f"   {label}: {conteo[label]:>5} clientes  ({pct[label]:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle("Distribución de Evasión de Clientes", fontsize=14, fontweight="bold")

# — Barras —
colores_bar = [COLORES[l] if l in (COLORES := {"No": AZUL, "Si": ROJO}) else AZUL
               for l in conteo.index]
bars = axes[0].bar(conteo.index, conteo.values, color=colores_bar,
                   edgecolor="white", linewidth=0.8, width=0.5)
for bar, val, p in zip(bars, conteo.values, pct.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 40, f"{val}\n({p:.1f}%)",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[0].set_title("Cantidad de Clientes", fontweight="bold")
axes[0].set_ylabel("N° de Clientes")
axes[0].set_ylim(0, conteo.max() * 1.18)
axes[0].tick_params(axis="x", labelsize=11)

# — Torta —
wedges, texts, autotexts = axes[1].pie(
    conteo.values,
    labels=conteo.index,
    colors=[AZUL, ROJO],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
    textprops=dict(fontsize=11),
)
for at in autotexts:
    at.set_fontweight("bold")
axes[1].set_title("Proporción de Evasión", fontweight="bold")

plt.tight_layout()
fig.savefig(f"{OUT}paso8_distribucion_evasion.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✅ Gráfico guardado: paso8_distribucion_evasion.png")


# ══════════════════════════════════════════════════════════════
# PASO 9 — EVASIÓN POR VARIABLES CATEGÓRICAS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 9: EVASIÓN POR VARIABLES CATEGÓRICAS")
print("=" * 60)

COLS_CAT = [
    ("Genero",          "Género"),
    ("Tipo_Contrato",   "Tipo de Contrato"),
    ("Metodo_Pago",     "Método de Pago"),
    ("Tipo_Internet",   "Tipo de Internet"),
    ("Adulto_Mayor",    "Adulto Mayor"),
    ("Factura_Digital", "Factura Digital"),
]

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
fig.suptitle("Tasa de Evasión por Variable Categórica", fontsize=15,
             fontweight="bold", y=1.01)

for ax, (col, titulo) in zip(axes.flat, COLS_CAT):
    # Calcular tasa de evasión por categoría
    tasa = (df.groupby(col)["Evasion"]
              .mean()
              .mul(100)
              .sort_values(ascending=False)
              .reset_index())
    tasa.columns = [col, "Tasa_Evasion"]

    barras = ax.barh(tasa[col].astype(str), tasa["Tasa_Evasion"],
                     color=[ROJO if v >= 30 else AZUL for v in tasa["Tasa_Evasion"]],
                     edgecolor="white", linewidth=0.6)

    for bar, val in zip(barras, tasa["Tasa_Evasion"]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", fontsize=9, fontweight="bold")

    ax.set_title(titulo, fontweight="bold", pad=8)
    ax.set_xlabel("Tasa de Evasión (%)")
    ax.set_xlim(0, tasa["Tasa_Evasion"].max() * 1.25)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

    # Imprimir tabla resumen en consola
    print(f"\n  {titulo}:")
    for _, row in tasa.iterrows():
        print(f"    {str(row[col]):<28} → {row['Tasa_Evasion']:.1f}%")

plt.tight_layout()
fig.savefig(f"{OUT}paso9_evasion_categoricas.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\n✅ Gráfico guardado: paso9_evasion_categoricas.png")


# ══════════════════════════════════════════════════════════════
# PASO 10 — EVASIÓN POR VARIABLES NUMÉRICAS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 10: EVASIÓN POR VARIABLES NUMÉRICAS")
print("=" * 60)

# Estadísticas por grupo
print("\n📊 Estadísticas por grupo (Evasion 0=No / 1=Sí):")
resumen = df.groupby("Evasion")[COLS_NUM].agg(["mean", "median", "std"]).round(2)
print(resumen.to_string())

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Distribución de Variables Numéricas según Evasión",
             fontsize=14, fontweight="bold", y=1.01)

for ax, col in zip(axes.flat, COLS_NUM):
    for label, color in [("No", AZUL), ("Si", ROJO)]:
        subset = df[df["Evasion_Label"] == label][col]
        sns.kdeplot(subset, ax=ax, color=color, fill=True,
                    alpha=0.35, linewidth=1.8, label=label)
        ax.axvline(subset.mean(), color=color, linestyle="--",
                   linewidth=1.2, alpha=0.8)

    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("")
    ax.legend(title="Evasión", fontsize=9)

plt.tight_layout()
fig.savefig(f"{OUT}paso10_evasion_numericas.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\n✅ Gráfico guardado: paso10_evasion_numericas.png")

# ── Boxplots comparativos ─────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Boxplots: Variables Numéricas por Evasión",
             fontsize=14, fontweight="bold")

for ax, col in zip(axes, COLS_NUM):
    sns.boxplot(data=df, x="Evasion_Label", y=col, ax=ax,
                order=["No", "Si"], hue="Evasion_Label",
                palette={"No": AZUL, "Si": ROJO}, legend=False,
                width=0.5, linewidth=1.2,
                flierprops=dict(marker="o", markersize=2, alpha=0.4))
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("Evasión")
    ax.set_ylabel("")

plt.tight_layout()
fig.savefig(f"{OUT}paso10_boxplots_numericas.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✅ Gráfico guardado: paso10_boxplots_numericas.png")

print("\n" + "=" * 60)
print("✅ ANÁLISIS EXPLORATORIO COMPLETADO")
print("=" * 60)

✅ Dataset listo — 7043 filas × 23 columnas

PASO 7: ANÁLISIS DESCRIPTIVO

📊 Estadísticas generales (DataFrame.describe()):
       Meses_Contrato  Cargo_Mensual  Cargo_Total  Cuentas_Diarias
count         7043.00        7043.00      7043.00          7043.00
mean            32.37          64.76      2281.91             2.16
std             24.56          30.09      2265.27             1.00
min              0.00          18.25        18.80             0.61
25%              9.00          35.50       402.22             1.18
50%             29.00          70.35      1394.55             2.35
75%             55.00          89.85      3786.60             3.00
90%             69.00         102.60      5973.69             3.42
max             72.00         118.75      8684.80             3.96

📊 Métricas adicionales:

  Meses_Contrato:
    Media:          32.37
    Mediana:        29.00
    Desv. Estándar: 24.56
    Asimetría:      0.24
    Curtosis:       -1.39
    Rango:          0.00 – 72.00

